# 🏭 Factory RAG: 매뉴얼 지식 검색 파이프라인 구축

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 임베딩 모델 로드 (가볍고 빠른 모델)
print("Loading sentence-transformer model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded.")

Loading sentence-transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded.


## 1. 공장 매뉴얼 텍스트 로드 및 분할 (Chunking)

In [2]:
# 텍스트 파일 읽기
with open('../data/cnc_manual.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# 간단하게 줄바꿈 단위로 문서를 쪼갭니다 (실무에서는 LangChain의 RecursiveCharacterTextSplitter 사용)
raw_chunks = text.split('\n')
documents = [doc.strip() for doc in raw_chunks if len(doc.strip()) > 10]

print(f"총 {len(documents)}개의 의미 있는 문장(Chunk)으로 분할되었습니다.\n")
for i, doc in enumerate(documents[:3]):
    print(f"Chunk {i}: {doc}")

총 13개의 의미 있는 문장(Chunk)으로 분할되었습니다.

Chunk 0: [DOCUMENT TYPE]: EQUIPMENT MANUAL & TROUBLESHOOTING GUIDE
Chunk 1: [EQUIPMENT]: CNC Milling Machine Model X-2000
Chunk 2: [DATE]: 2024-01-15


## 2. 벡터화 및 FAISS 인덱스 구축

In [3]:
# 문서를 벡터로 임베딩
print("문서 임베딩 중...")
doc_embeddings = embedder.encode(documents)

# FAISS 벡터 인덱스 생성
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings, dtype=np.float32))

print(f"FAISS 벡터 DB에 {index.ntotal}개의 제조 매뉴얼 청크가 적재되었습니다.")

문서 임베딩 중...
FAISS 벡터 DB에 13개의 제조 매뉴얼 청크가 적재되었습니다.


## 3. Retriever 함수 정의 (Agent가 사용할 도구)

In [4]:
def search_manual(query: str, k: int = 1) -> str:
    """
    질문과 가장 관련성 높은 공장 매뉴얼 내용을 찾아 반환합니다.
    """
    query_vector = embedder.encode([query])
    distances, indices = index.search(np.array(query_vector, dtype=np.float32), k)
    
    retrieved_doc = documents[indices[0][0]]
    return f"[Manual DB] 검색된 내용: {retrieved_doc}"

# 테스트
test_query = "E-04 에러의 원인이 무엇인가요?"
print(f"사용자 질문: {test_query}")
print(search_manual(test_query))

사용자 질문: E-04 에러의 원인이 무엇인가요?
[Manual DB] 검색된 내용: [2024-08-22] Machine #1 had low coolant (E-01). Added 5L of synthetic coolant.
